# 인사팀 의도 분류 평가 — Colab Pro+ · Qwen3.5-27B

인사팀(hr_admin) 질문을 **규칙**으로 분류할 때와 **LLM(Qwen3.5-27B, vLLM)** 으로 분류할 때(hybrid)의 점수를 비교해요.
골든셋(`admin_golden_set.json`)으로 의도·집계 도구·도구에 넘긴 값을 채점하고, **답변 생성은 안 해요.**

- 런타임: **H100 또는 A100**. L4·T4는 메모리가 부족해서 27B가 안 올라가요.
- 위에서부터 차례로 실행해요. 처음엔 모델 다운로드가 20~40분 걸려요.
- 비교 방식: `rule`(규칙만) ↔ `hybrid`(규칙이 못 정한 문장만 LLM). 코드는 `backend/scripts/eval_admin_intent.py`를 그대로 써요.

## 1. 설정

In [ ]:
CONFIG = {
    "code_source": "local",   # "git": GitHub에서 backend를 받아요 / "local": /content/hr_chatbot/backend 가 이미 있을 때 그대로 써요
    "git_branch": "main",     # code_source가 "git"일 때 받을 backend 브랜치
    "modes": ["rule", "hybrid"],
    "split": "all",           # all / dev / test
    "model": None,            # None이면 GPU 메모리를 보고 자동 선택. 직접 정하려면 예: "Qwen/Qwen3.5-27B-FP8"
}

## 2. GPU 확인과 모델 고르기

In [ ]:
import os, sys, subprocess, json, time, urllib.request
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("GPU가 없어요. 런타임 → 런타임 유형 변경에서 H100 또는 A100을 고르고 다시 실행하세요.")
gpu_name, mem_gb = gpu.split(",")[0].strip(), int(gpu.split(",")[1]) / 1024
if CONFIG["model"]:
    MODEL, MAX_LEN = CONFIG["model"], 8192
elif mem_gb >= 70:
    MODEL, MAX_LEN = "Qwen/Qwen3.5-27B", 16384          # H100 · A100 80GB (bf16, 약 54GB)
elif mem_gb >= 36:
    MODEL, MAX_LEN = "Qwen/Qwen3.5-27B-FP8", 8192       # A100 40GB (FP8, 약 28GB)
else:
    raise RuntimeError(f"{gpu_name} {mem_gb:.0f}GB는 메모리가 부족해서 27B를 못 올려요. H100 또는 A100으로 바꾸세요.")
SERVED = "qwen3.5-27b"
print(f"GPU {gpu_name} {mem_gb:.0f}GB → {MODEL} (문맥 {MAX_LEN})")

## 3. vLLM 서버 띄우기

`HF_TOKEN`이 보안 비밀(왼쪽 열쇠 아이콘)에 있으면 모델을 더 빨리 받아요. 없어도 돌아가요.
Qwen3.5는 Mamba 층이 있어서 `--max-num-seqs 16`이 필요해요 (기본 256이면 시작이 안 될 수 있어요).

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN 사용")
except Exception:
    print("HF_TOKEN 없음 (그냥 진행)")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm", "httpx"], check=True)
help_text = subprocess.run(["vllm", "serve", "--help=all"], capture_output=True, text=True).stdout
extra = ["--default-chat-template-kwargs", '{"enable_thinking": false}'] if "--default-chat-template-kwargs" in help_text else []
print("생각 모드 끔" if extra else "생각 모드 끄기 옵션 없음 → 요청마다 chat_template_kwargs로 끔")
vllm = subprocess.Popen(
    ["vllm", "serve", MODEL, "--served-model-name", SERVED, "--max-model-len", str(MAX_LEN),
     "--gpu-memory-utilization", "0.90", "--max-num-seqs", "16", "--port", "8000", *extra],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

for _ in range(150):    # 첫 실행은 모델 다운로드로 최대 40분 이상 걸려요
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5)
        print("vLLM 준비됨"); break
    except Exception:
        if vllm.poll() is not None:
            log = open("/content/vllm.log", encoding="utf-8", errors="replace").read().splitlines()
            keys = ("out of memory", "OutOfMemory", "Error:", "ValueError", "not supported", "No available memory", "max_model_len", "CUDA")
            print("── 원인 후보 ──"); print("\n".join([l for l in log if any(k in l for k in keys)][-25:]) or "(없음)")
            raise RuntimeError("vLLM이 시작하지 못했어요. 위 원인 후보를 확인하세요 (전체 로그: /content/vllm.log)")
        time.sleep(20)
else:
    raise RuntimeError("vLLM이 50분 안에 준비되지 않았어요. /content/vllm.log를 확인하세요")

## 4. backend 코드 가져오기

`code_source`가 `"git"`이면 GitHub에서 받아요. 골든셋은 backend에 없어서 RAG 저장소에서 복사해 와요 (커밋 대상이 아니에요).

In [ ]:
ROOT = Path("/content/hr_chatbot"); ROOT.mkdir(parents=True, exist_ok=True)
BACKEND = ROOT / "backend"
URL = "https://github.com/Jaeyong-s-HR-Chatbot/{}.git"

def sh(*cmd, **kw):
    return subprocess.run(cmd, check=True, **kw)

if CONFIG["code_source"] == "git":
    if not BACKEND.is_dir():
        sh("git", "clone", "-q", "--depth", "1", "-b", CONFIG["git_branch"], URL.format("backend"), str(BACKEND))
    else:
        sh("git", "-C", str(BACKEND), "fetch", "-q", "--depth", "1", "origin", CONFIG["git_branch"])
        sh("git", "-C", str(BACKEND), "checkout", "-q", "-B", CONFIG["git_branch"], "FETCH_HEAD")
elif not BACKEND.is_dir():
    raise RuntimeError(f"{BACKEND} 가 없어요. CONFIG의 code_source를 'git'으로 바꾸세요.")
head = subprocess.run(["git", "-C", str(BACKEND), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("backend:", head)

if not (ROOT / "RAG").is_dir():
    sh("git", "clone", "-q", "--depth", "1", URL.format("RAG"), str(ROOT / "RAG"))
(BACKEND / "eval/golden").mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy(ROOT / "RAG/eval/golden/admin_golden_set.json", BACKEND / "eval/golden/")
print("RAG:", subprocess.run(["git", "-C", str(ROOT / "RAG"), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

sh(sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND / "requirements.txt"))
golden = json.loads((BACKEND / "eval/golden/admin_golden_set.json").read_text(encoding="utf-8"))
n_admin = sum(1 for i in golden["items"] if i["role"] == "hr_admin")
print(f"인사팀 문항 {n_admin}개 (골든셋 전체 {len(golden['items'])}개)")

## 5. LLM 연결

backend의 `admin_llm_classify._request`는 Ollama 형식이에요. 여기서 vLLM(OpenAI 형식) 요청으로 바꿔 끼워요. 프롬프트·검증 코드는 그대로예요.

In [ ]:
os.environ.update({"LLM_BASE_URL": "http://127.0.0.1:8000/v1", "LLM_MODEL": SERVED, "LLM_API_KEY": "EMPTY",
                   "CLASSIFY_MODEL": SERVED, "CLASSIFY_TIMEOUT": "300", "HRDB_URL": "", "CHAT_LOG_DB": "false",
                   "PSEUDO_KEY": "colab-eval"})
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
if str(BACKEND / "scripts") not in sys.path:
    sys.path.insert(0, str(BACKEND / "scripts"))

import httpx
from app import config
from app.graph import admin_llm_classify as alc

def vllm_request(question, today):
    body = {"model": SERVED,
            "messages": [{"role": "system", "content": alc._system_prompt(today)}, {"role": "user", "content": question}],
            "temperature": 0, "max_tokens": 1024,
            "chat_template_kwargs": {"enable_thinking": False}}
    r = httpx.post("http://127.0.0.1:8000/v1/chat/completions", timeout=300,
                   json={**body, "response_format": {"type": "json_schema",
                                                    "json_schema": {"name": "admin_intent", "schema": alc.OUTPUT_SCHEMA}}})
    if r.status_code == 400:    # 스키마를 못 받으면 프롬프트의 JSON 지시만으로 다시 요청
        r = httpx.post("http://127.0.0.1:8000/v1/chat/completions", json=body, timeout=300)
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]
    if not isinstance(content, str):
        raise TypeError("vLLM response content is not text")
    return content

alc._request = vllm_request
test_q = "올해 연차 사용률 50% 안 되는 부서 어디야?"
print("시험 호출:", alc.classify(test_q, __import__("datetime").date(2026, 11, 9)))

## 6. 평가 실행

In [ ]:
import eval_admin_intent as ev
t0 = time.time()
result = ev.run(CONFIG["modes"], CONFIG["split"])
print(f"완료 ({time.time() - t0:.0f}초)")

## 7. 채점

In [ ]:
from IPython.display import Markdown, display
display(Markdown(ev.markdown(result)))
for mode, r in result.items():
    print(mode, "LLM:", r["llm"])

In [ ]:
import pandas as pd

def wrong(x):
    return x["scored"] and (not x["intent_ok"] or x["tool_ok"] is False or (x["tool_ok"] and x["params_ok"] is False))

changes = pd.DataFrame()
if "rule" in result and "hybrid" in result:
    rule = {x["id"]: x for x in result["rule"]["rows"]}
    rows = []
    for h in result["hybrid"]["rows"]:
        r = rule[h["id"]]
        if wrong(r) == wrong(h) and (r["pred_intent"], r["pred_tool"]) == (h["pred_intent"], h["pred_tool"]):
            continue
        kind = "LLM이 고침" if wrong(r) and not wrong(h) else "새로 틀림" if wrong(h) and not wrong(r) else "그대로 틀림(답이 바뀜)" if wrong(h) else "맞음(답이 바뀜)"
        rows.append({"결과": kind, "id": h["id"], "질문": h["question"], "정답": f"{h['gold_intent']}/{h['gold_tool']}",
                     "rule": f"{r['pred_intent']}/{r['pred_tool']}", "hybrid": f"{h['pred_intent']}/{h['pred_tool']}"})
    changes = pd.DataFrame(rows).sort_values(["결과", "id"]) if rows else pd.DataFrame(columns=["결과", "id", "질문", "정답", "rule", "hybrid"])
    print("rule → hybrid 에서 바뀐 문항:", changes["결과"].value_counts().to_dict())
    display(changes)

## 8. 결과 저장

`/content/admin_intent_results/`에 저장하고 zip으로 내려받아요. 그 zip을 공유해 주세요.

In [ ]:
import datetime as dt
out_dir = Path("/content/admin_intent_results") / f"admin_intent_{dt.datetime.now():%Y%m%d_%H%M}_{SERVED}"
ev.save(result, SERVED, out_dir)
changes.to_csv(out_dir / "rule_vs_hybrid_changes.csv", index=False, encoding="utf-8-sig")
(out_dir / "env.json").write_text(json.dumps({"model": MODEL, "gpu": gpu_name, "backend": head, "split": CONFIG["split"],
                                             "modes": CONFIG["modes"], "인사팀 문항": n_admin}, ensure_ascii=False, indent=1), encoding="utf-8")
zip_path = shutil.make_archive(str(out_dir), "zip", out_dir)
print("저장:", out_dir, "\nzip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("자동 다운로드 실패 (왼쪽 파일 창에서 직접 받으세요):", type(e).__name__)

## 9. 런타임 종료 (끝났을 때만)

결과 zip을 받은 뒤에 실행하세요. vLLM을 끄고 런타임을 반납해서 컴퓨트 유닛 소모를 멈춰요.

In [ ]:
vllm.terminate()
from google.colab import runtime
runtime.unassign()